<a href="https://colab.research.google.com/github/mayajdias/ds2002-fa26/blob/main/notebooks/01-foundations/2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Total units: {total_units}")

Total revenue: $8,520.00
Total units: 783


The 400 orders generated a total of $8,520.00 in revenue from 783 units sold. Total revenue falls between the 8000-9000 range. The 783 units sold means the average order was just under 2 units (783/400).

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = (
    df.groupby('category', as_index=False)
      .agg(revenue=('revenue', 'sum'))
)

by_category['share_pct'] = by_category['revenue'] / total_revenue * 100

by_category = by_category.sort_values('revenue', ascending=False)

by_category['share_pct'] = by_category['share_pct'].round(1)

print(by_category)

   category  revenue  share_pct
1      Food   4293.0       50.4
2     Merch   1771.5       20.8
0     Drink   1554.0       18.2
3  RainGear    901.5       10.6


Food generated the most revenue at \$4,293.00, accounting for 50.4% of total revenue. Merch was second at \$1,771.50, or 20.8%, followed by Drink at 18.2% and RainGear at 10.6%.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
by_vendor = (
    df.groupby('vendor_id', as_index=False)
      .agg(
          avg_order_revenue=('revenue', 'mean'),
          order_count=('revenue', 'size')
      )
      .sort_values('avg_order_revenue', ascending=False)
)

by_vendor['avg_order_revenue'] = by_vendor['avg_order_revenue'].round(2)

print(by_vendor)

  vendor_id  avg_order_revenue  order_count
0      V-01              22.60           94
3      V-18              21.75          108
1      V-05              20.58           93
2      V-10              20.31          105


V-01 had the highest average order revenue at \$22.60 per order across 94 orders. V-18 was close behind at \$21.75 across 108 orders. Because all four vendors have between 93 and 108 orders, the comparison is not being driven by an unusually small sample.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_share = (
    df.loc[df['category'] == 'Merch', 'revenue'].sum()
    / df['revenue'].sum()
    * 100
)

print(f"Merch share of revenue: {merch_share:.1f}%")

Merch share of revenue: 20.8%


Merch accounts for 20.8% of total revenue, meaning a little more than one-fifth of all revenue from the 400 orders came from merchandise sales.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:


vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

original_rows = len(df)
original_revenue = df['revenue'].sum()

joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

unmatched = joined.loc[
    joined['vendor_name'].isna(),
    'vendor_id'
].unique()

print("Unmatched vendor(s):", unmatched)
print("Rows before merge:", original_rows)
print("Rows after merge:", len(joined))
print(f"Revenue before merge: ${original_revenue:,.2f}")
print(f"Revenue after merge: ${joined['revenue'].sum():,.2f}")

# Keep the unmatched orders but give them a readable placeholder name
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown Vendor (V-18)')

Unmatched vendor(s): ['V-18']
Rows before merge: 400
Rows after merge: 400
Revenue before merge: $8,520.00
Revenue after merge: $8,520.00


**The unmatched vendor, and what I did about it:** The unmatched vendor was V-18. I kept its orders because this was a left join and labeled it "Unknown Vendor (V-18)" rather than dropping the observations. The merge preserved all 400 rows and the original $8,520.00 in revenue, so no order or revenue data was lost.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

print(pivot)

category                Drink    Food   Merch  RainGear   Total
vendor_name                                                    
Cav Merch North         502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers            171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos           298.5   882.0   489.0     244.5  1914.0
Unknown Vendor (V-18)   582.0  1018.5   508.5     240.0  2349.0
Total                  1554.0  4293.0  1771.5     901.5  8520.0


The pivot table shows that Unknown Vendor (V-18) generated the highest overall revenue at \$2,349.00, while Rotunda Tacos generated the lowest at \$1,914.00. Food was the largest category overall at \$4,293.00, and the row and column totals reconcile to total revenue of \$8,520.00.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) For the next game, I would recommend that vendors prioritize food inventory while also making sure they have enough merchandise and drinks available. Food generated \$4,293.00, which was 50.4% of the total \$8,520.00 in revenue, making it by far the strongest category. Merch was the second-largest category at \$1,771.50, or 20.8% of revenue, while drinks generated another \$1,554.00. These results suggest that vendors should devote the most inventory and preparation capacity to food but should not overlook merchandise and drinks, which together represent a substantial share of sales. Vendors could also look at their own category mix before the next game to identify areas where their sales are weaker than the overall demand pattern.

b) The vendor-level report is the least trustworthy because the vendor-name lookup is incomplete. V-18 appears in 108 orders but does not have a matching vendor name, so I had to label it "Unknown Vendor (V-18)." Its revenue values are still included correctly, but I would be cautious about making recommendations to a specific named business based on the vendor comparison until V-18's actual identity is confirmed.
